# WEEK 5 — End-to-End RAG Fundamentals
**Goal:** build a simple end-to-end RAG notebook from the 3 PDF files in this folder.

## End-to-end RAG steps
1. Load the source documents from the folder.
2. Split the documents into chunks that preserve meaning.
3. Convert chunks into embeddings.
4. Store embeddings in a vector database.
5. Retrieve the most relevant chunks for a question.
6. Ask the LLM to answer using only the retrieved context.
7. Review the result and improve the previous steps.

## How we will enhance each step
- Step 1: check file names, page counts, and page previews.
- Step 2: compare chunk size and overlap.
- Step 3: compare embedding model choices.
- Step 4: see how the vector store changes retrieval.
- Step 5: test different questions.
- Step 6: improve the answer prompt and citation style.

## Why this version is simpler
- one PDF folder only
- one embedding model only
- one chunking strategy only
- direct code that is easier for students to follow

---

**Notebook annotations:** After each code cell there is a short markdown explanation describing what each import, function call, or print statement does and why it is included. Run the notebook from top to bottom; when a cell fails, read the explanation cell immediately below it to understand the intent and required environment or inputs.

## Embeddings
We use OpenAI embeddings because they are easy to understand and strong for RAG.

**Types to know**
- `text-embedding-3-small`: cheapest and fast; good default for most RAG demos.
- `text-embedding-3-large`: stronger semantic quality; use when retrieval accuracy matters more than cost.
- `text-embedding-ada-002`: older legacy model; useful mainly for compatibility with older projects.

**Important difference**
- smaller and larger models both turn text into vectors, but the larger model usually captures meaning better and costs more.
- the 3-series also supports dimensionality control, which can reduce storage size when needed.

## Step 0 — Environment check
Before we start, confirm the notebook can see the local `.env` file, the 3 PDF files in this folder, and the LangSmith keys that power tracing.

**Enhance this step**
- verify the OpenAI key is present
- verify the LangSmith key is present
- turn on tracing for LangSmith
- confirm the notebook is running from the expected folder

In [32]:
# Environment check
from pathlib import Path
import os
from dotenv import load_dotenv

env_path = Path.cwd() / '.env'
load_dotenv(env_path, override=True)

os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ.setdefault('LANGCHAIN_PROJECT', os.getenv('LANGCHAIN_PROJECT', 'langsmith_demo'))
if os.getenv('LANGSMITH_API_KEY'):
    os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

print('CWD:', Path.cwd())
print('Loaded .env:', env_path.exists())
print('OPENAI_API_KEY present:', bool(os.getenv('OPENAI_API_KEY')))
print('LANGSMITH_API_KEY present:', bool(os.getenv('LANGSMITH_API_KEY')))
print('LANGSMITH tracing:', os.getenv('LANGSMITH_TRACING'))
print('LANGCHAIN project:', os.getenv('LANGCHAIN_PROJECT'))
print('PDF files in folder:', [p.name for p in sorted(Path.cwd().glob('*.pdf'))])

CWD: d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 6 — RAG Optimization + LLM Evaluation + LangSmith\Live
Loaded .env: True
OPENAI_API_KEY present: True
LANGSMITH_API_KEY present: True
LANGSMITH tracing: true
LANGCHAIN project: langsmith_demo
PDF files in folder: ['HDFC-Life-Group-Term-Life-Policy.pdf', 'HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf', 'HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf']


In [33]:
# Packages used in this notebook
# - python-dotenv
# - langchain-community
# - langchain-text-splitters
# - langchain-openai
# - chromadb

## Step 1 — Load the PDFs
This notebook uses the 3 PDF files in this folder as the source documents.

**Enhance this step**
- print the filenames
- inspect the number of pages loaded
- preview the first page so students can see the raw text that enters the pipeline

In [34]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_files = sorted(Path.cwd().glob('*.pdf'))
docs = []
for pdf_file in pdf_files:
    docs.extend(PyPDFLoader(str(pdf_file)).load())

print('PDF files:', [p.name for p in pdf_files])
print('Loaded pages:', len(docs))
print('First page preview:')
print(docs[0].page_content[:800].replace('\n', ' '))

PDF files: ['HDFC-Life-Group-Term-Life-Policy.pdf', 'HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf', 'HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf']
Loaded pages: 101
First page preview:
F&U dated 15th October 2022                  UIN-101N169V02  P a g e  | 0                                      HDFC Life Group Term Life    OF      «OWNERNAME»               Based on the Proposal and the declarations and  any  statement made or referred to therein,  We will pay the Benefits mentioned in this Policy  subject to the terms and conditions contained  herein              << Designation of the Authorised Signatory >>


## Step 2 — Chunk the PDF pages
The chunk size controls how much text goes into each embedding, and overlap keeps nearby context together.

**Enhance this step**
- try a larger chunk size for long legal-style paragraphs
- try a smaller chunk size for precise retrieval
- change overlap to see how much neighboring context is preserved

In [63]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)
chunks = splitter.split_documents(docs)

print('Chunks created:', len(chunks))
print('First chunk preview:')
print(chunks[0].page_content[:800].replace('\n', ' '))

Chunks created: 609
First chunk preview:
F&U dated 15th October 2022                  UIN-101N169V02  P a g e  | 0                                      HDFC Life Group Term Life    OF      «OWNERNAME»               Based on the Proposal and the declarations and  any  statement made or referred to therein,  We will pay the Benefits mentioned in this Policy  subject to the terms and conditions contained  herein              << Designation of the Authorised Signatory >>


## Step 3 — Build embeddings and the vector store
We use `text-embedding-3-small` here because it is a strong default for a classroom RAG demo.

**Enhance this step**
- compare `text-embedding-3-small` and `text-embedding-3-large`
- notice how stronger embeddings can improve retrieval quality
- keep the vector store persistent so students can re-open it later

In [64]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.messages import SystemMessage, HumanMessage

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='rag_chroma_store',
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 6})

print('Vector store ready:', len(chunks), 'chunks')

Vector store ready: 609 chunks


In [65]:
# Step 3 — answer the question directly, line by line
from langchain_openai import ChatOpenAI  # import the chat model used to generate the answer
from langchain_core.messages import SystemMessage, HumanMessage  # import message types for the prompt

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)  # create the chat model used to generate the answer

question = 'What benefits are described in the documents?'  # ask one sample question
context_docs = retriever.invoke(question)  # AUGMENT: retrieve the most relevant chunks

context_text = '\n\n'.join(  # build the context block passed into the model
    f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
    for doc in context_docs
)

messages = [  # prepare the prompt messages for generation
    SystemMessage(content='You are a helpful tutor. Answer only from the provided context and keep the answer clear and student-friendly.'),
    HumanMessage(content=f'Context:\n{context_text}\n\nQuestion: {question}\n\nAnswer with short citations to the source file names.'),
]

response = llm.invoke(messages)  # GENERATE: ask the model to write the answer

print('Retrieved context preview:')  # show the augmentation stage
for doc in context_docs:  # list each retrieved chunk before the answer
    source_name = doc.metadata.get('source', 'unknown')  # get the source name
    preview = doc.page_content[:220].replace('\n', ' ')  # keep the preview short and readable
    print('-', source_name)  # show the source file
    print(' ', preview + ('...' if len(doc.page_content) > 220 else ''))  # show a short chunk preview
    print()  # blank line between chunks

print('Final answer:')  # ANSWER: print the final response
print(response.content)  # show the generated answer


Retrieved context preview:
- d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 6 — RAG Optimization + LLM Evaluation + LangSmith\Live\HDFC-Life-Group-Term-Life-Policy.pdf
  15. Benefits means the Benefit as mentioned in Part C of this Policy Document.    16. Benefit Expiry Age means the Age in Years last birthday as mentioned in the Policy Schedule.    17. Certificate of Insurance in respec...

- d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 6 — RAG Optimization + LLM Evaluation + LangSmith\Live\HDFC-Life-Group-Term-Life-Policy.pdf
  13. Basic Policy means and includes this document, Coverage Schedule, the signed Proposal Form,  the Policy Schedule and any attached endorsements or supplements together with all addendums.     14. Beneficiary shall mea...

- d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 6 — RAG Optimization + LLM Evaluation + LangSmith\Live\HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf
  utilization of the monies so paid.    C.6.3.  Apart fr

## Step 4 — Retrieve the best chunks
The retriever picks the chunks that are most relevant to the user question.

**Enhance this step**
- test whether `k=3`, `k=4`, or `k=6` gives better context
- inspect the retrieved chunks before generating the answer
- check whether the retrieved text comes from the right PDF

In [66]:
# Step 5 — try a few test questions and compare the retrieved answers
# Enhancement: use harder questions to see where the retriever misses context.
from langchain_core.messages import SystemMessage, HumanMessage

if not ('retriever' in globals() and 'llm' in globals()):
    print('Run the previous cells first so the retriever and LLM are ready.')
else:
    test_questions = [
        'What benefits are described in the documents?',
        'What is the policy term or duration?',
        'What happens on maturity or death?',
    ]

    for question in test_questions:
        print('=' * 90)
        print('Question:', question)
        # AUGMENT: retrieve the most relevant chunks
        context_docs = retriever.invoke(question)
        # build the context string
        context_text = '\n\n'.join(
            f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
            for doc in context_docs
        )
        # GENERATE: create messages and call the LLM
        messages = [
            SystemMessage(content='You are a helpful tutor. Answer only from the provided context and keep the answer clear and student-friendly.'),
            HumanMessage(content=f'Context:\n{context_text}\n\nQuestion: {question}\n\nAnswer with short citations to the source file names.'),
        ]
        response = llm.invoke(messages)
        # show retrieved context preview
        print('Retrieved context preview:')
        for doc in context_docs:
            source_name = doc.metadata.get('source', 'unknown')
            preview = doc.page_content[:220].replace('\n', ' ')
            print('-', source_name)
            print(' ', preview + ('...' if len(doc.page_content) > 220 else ''))
            print()
        # show final answer
        print('Final answer:')
        print(response.content)


Question: What benefits are described in the documents?
Retrieved context preview:
- d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 6 — RAG Optimization + LLM Evaluation + LangSmith\Live\HDFC-Life-Group-Term-Life-Policy.pdf
  15. Benefits means the Benefit as mentioned in Part C of this Policy Document.    16. Benefit Expiry Age means the Age in Years last birthday as mentioned in the Policy Schedule.    17. Certificate of Insurance in respec...

- d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 6 — RAG Optimization + LLM Evaluation + LangSmith\Live\HDFC-Life-Group-Term-Life-Policy.pdf
  13. Basic Policy means and includes this document, Coverage Schedule, the signed Proposal Form,  the Policy Schedule and any attached endorsements or supplements together with all addendums.     14. Beneficiary shall mea...

- d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 6 — RAG Optimization + LLM Evaluation + LangSmith\Live\HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf


# Class Exercises — Try in class

### Exercise 1 — Chunking experiment
**Task:** Change chunk size and overlap, rebuild the vector store, and compare retrieved chunks for a sample question. Try at least two settings (e.g., chunk_size=800, overlap=200 and chunk_size=400, overlap=100) and note differences in which chunks are retrieved.

**Solution & explanation:**
- Why: Smaller chunks increase precision (less unrelated text per chunk) while larger chunks increase context coverage. Overlap increases recall by duplicating boundary content.
- What to observe: which PDF/file the top chunks come from, and whether the retrieved text contains the explicit answer.

**Solution (example code):**
```python
# Create a new splitter with larger chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
new_chunks = splitter.split_documents(docs)  # docs = list of original Document objects

# Rebuild the vectorstore with the same embeddings
new_vectorstore = Chroma.from_documents(new_chunks, embeddings=embeddings)
new_retriever = new_vectorstore.as_retriever(search_kwargs={"k": 4})

# Query and inspect retrieved chunks
q = "What benefits are described in the documents?"
context_docs = new_retriever.get_relevant_documents(q)
for d in context_docs:
    print('-', d.metadata.get('source', 'unknown'))
    print(d.page_content[:300].replace('\n', ' '))
    print()

# Use the same answer function (it will use the retriever variable if you swap it in),
# or call your generation step with context_docs as done earlier.
```

### Exercise 2 — Hybrid tuning (lexical + dense)
**Task:** Implement a simple fusion of TF-IDF scores and dense scores (alpha-weighted) and test alpha values [0.2, 0.5, 0.8]. Report which alpha works best for 3 sample queries.


### Exercise 3 — Re-ranking and final answer quality
**Task:** Take the top-5 candidates from your retriever and re-rank them using a Cross-Encoder (or a lightweight dense dot-product re-ranker). Compare the final answer quality (before/after re-rank) for a sample question.

**Solution & explanation:**
- Why: The retriever is optimized for speed and recall; the Cross-Encoder uses joint scoring of (query, passage) pairs and often improves ranking precision at the cost of latency.
- Expected result: the true answer passage should move up in ranking after re-ranking if the re-ranker is effective.

**Step-by-step code for students:**
```python
# Step 1: create the chat model that will generate the final answer
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Step 2: ask a question
question = 'What benefits are described in the documents?'

# Step 3: retrieve the most relevant chunks
context_docs = retriever.invoke(question)

# Step 4: build one context string from the retrieved chunks
context_text = '\n\n'.join(
    f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
    for doc in context_docs
)

# Step 5: create the messages for the model
messages = [
    SystemMessage(content='You are a helpful tutor. Answer only from the provided context and keep the answer clear and student-friendly.'),
    HumanMessage(content=f'Context:\n{context_text}\n\nQuestion: {question}\n\nAnswer with short citations to the source file names.'),
]

# Step 6: ask the model to generate the answer
response = llm.invoke(messages)

# Step 7: print the retrieved context preview
print('Retrieved context preview:')
for doc in context_docs:
    source_name = doc.metadata.get('source', 'unknown')
    preview = doc.page_content[:220].replace('\n', ' ')
    print('-', source_name)
    print(' ', preview + ('...' if len(doc.page_content) > 220 else ''))
    print()

# Step 8: print the final answer
print('Final answer:')
print(response.content)
```

**Why this version is easier to learn:**
- Each line shows one part of the RAG flow.
- Students can run and inspect the result after every step.
- It separates augment, generate, and answer into visible stages.

---

**Class instructions:**
- Split students into small groups, assign each exercise, let them implement, then present results. Discuss tradeoffs and what changed in retrieved context and final answer quality.


### Exercise 3 — Re-ranking and final answer quality
**Task:** Take the top-5 candidates from your retriever and re-rank them using a Cross-Encoder (or a lightweight dense dot-product re-ranker). Compare the final answer quality (before/after re-rank) for a sample question.

**Solution & explanation:**
- Why: The retriever is optimized for speed and recall; the Cross-Encoder uses joint scoring of (query, passage) pairs and often improves ranking precision at the cost of latency.
- Expected result: the true answer passage should move up in ranking after re-ranking if the re-ranker is effective.

**Solution (example code):**
```python
# Option A: Cross-Encoder (may download model, runs slower)
from sentence_transformers import CrossEncoder
ce = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

candidates = hybrid_retrieve(q, top_k=5, alpha=0.6)
pairs = [(q, c[2]) for c in candidates]
scores = ce.predict(pairs)
ranked = sorted(zip([c[0] for c in candidates], scores, [c[2] for c in candidates]), key=lambda x: x[1], reverse=True)
print('Re-ranked:')
for i, s, t in ranked:
    print(i, s, t[:200])

# Option B: Lightweight dot-product re-ranker using the same SVD proxy (no model download)
# q_emb = svd.transform(dense_vectorizer.transform([q])); normalize
# cand_embs = emb_docs[[idx for idx,_,_ in candidates]]  # emb_docs were normalized when added
# sims = cand_embs @ q_emb[0]
# sort by sims and display
```

---

**Class instructions:**
- Split students into small groups, assign each exercise, let them implement, then present results. Discuss tradeoffs and what changed in retrieved context and final answer quality.

## Step 6 — Review the answers and improve the pipeline
Use the test questions above to decide whether to improve chunking, embeddings, retrieval `k`, or the final answer prompt.

**Enhance this step**
- change chunk size or overlap and re-run the questions
- compare the retrieved context for each answer
- adjust the prompt so the model cites sources more clearly

In [67]:
evaluation_set = [
    {
        "question": "What is the free look period in group term policy?",
        "gold_sources": ["HDFC-Life-Group-Term-Life-Policy"]
    },
    {
        "question": "Explain maturity benefit in HDFC Life Sanchay Plus policy",
        "gold_sources": ["HDFC-Life-Sanchay-Plus"]
    },
    {
        "question": "What is death benefit in Sampoorna Jeevan policy?",
        "gold_sources": ["HDFC-Life-Sampoorna-Jeevan"]
    }
]

### Explanation: Evaluation set
- `evaluation_set` is a small labeled dataset mapping sample questions to the expected source filenames (gold sources).
- Use this to measure retrieval accuracy (precision@k, recall@k) by checking whether the retriever returns chunks from the expected documents.
- Keep filenames concise and consistent with `Document.metadata['source']` values used by your loader.

In [68]:
evaluation_set += [
    {
        "question": "What is grace period in Sampoorna Jeevan?",
        "gold_sources": ["HDFC-Life-Sampoorna-Jeevan"]
    },
    {
        "question": "What happens if premium is not paid in Sanchay Plus?",
        "gold_sources": ["HDFC-Life-Sanchay-Plus"]
    },
    {
        "question": "What is sum assured on death in Sampoorna Jeevan?",
        "gold_sources": ["HDFC-Life-Sampoorna-Jeevan"]
    }
]

In [69]:
# Evaluate retrieval accuracy
# - Use a small labeled set mapping questions -> gold source file names or passage ids.
# - Compute precision@k (how many of top-k contain the gold source).

def precision_at_k(retriever, question, gold_sources, k=4):
    docs = retriever.invoke(question)
    topk = docs[:k]
    # compare against the provided gold source names or ids
    hits = sum(1 for d in topk if any(gs.lower() in d.metadata.get('source', '').lower() for gs in gold_sources))
    return hits / k

# Example usage:
# gold = ['policy.pdf']
# print('Precision@4:', precision_at_k(retriever, 'What benefits are described?', gold, k=4))

### Explanation: `precision_at_k` function
- Inputs: `retriever`, `question`, `gold_sources`, and `k` (top-k to consider).
- `retriever.invoke(question)` returns a ranked list of Document objects; we take the first `k` results.
- We count a hit when any gold source substring matches the document's `metadata['source']` (case-insensitive).
- The function returns `hits / k` (precision at `k`). Use it to compare retrievers or settings.

### Evaluation — Precision@k & Recall@k
- This section computes Precision@k and Recall@k for a small labeled evaluation set.
- Precision@k: of the top-k returned documents, what fraction are from the gold sources (measures precision).
- Recall@k: whether at least one gold source appears in the top-k (measures coverage).
- Use both metrics together to understand trade-offs: high recall with low precision suggests noisy results; high precision with low recall suggests missing relevant passages.

In [ ]:
results_precision = []
results_recall = []

for item in evaluation_set:
    question = item["question"]
    gold = item["gold_sources"]

    p_score = precision_at_k(retriever, question, gold, k=4)
    r_score = recall_at_k(retriever, question, gold, k=4)

    print("=" * 60)
    print("Question:", question)
    print("Precision@4:", p_score)
    print("Recall@4:", r_score)

    results_precision.append(p_score)
    results_recall.append(r_score)

# Average scores
avg_precision = sum(results_precision) / len(results_precision) if results_precision else 0.0
avg_recall = sum(results_recall) / len(results_recall) if results_recall else 0.0
print("\nAverage Precision@4:", avg_precision)
print("Average Recall@4:", avg_recall)


Question: What is the free look period in group term policy?
Precision@4: 0.0
Question: Explain maturity benefit in HDFC Life Sanchay Plus policy
Precision@4: 1.0
Question: What is death benefit in Sampoorna Jeevan policy?
Precision@4: 1.0
Question: What is grace period in Sampoorna Jeevan?
Precision@4: 0.75
Question: What happens if premium is not paid in Sanchay Plus?
Precision@4: 1.0
Question: What is sum assured on death in Sampoorna Jeevan?
Precision@4: 1.0

Average Precision@4: 0.7916666666666666


### Explanation: Running the evaluation loop
- Iterate over each labeled example in `evaluation_set`, compute precision@4, print the per-question score, and then compute the average.
- Use this loop to quickly benchmark how well your retriever returns the expected documents under current chunking/embedding settings.
- If the score is low, consider adjusting chunk size, embedding model, or retriever `k` and re-run.

## Final note
This notebook now shows the full RAG flow in order, and each stage has a small enhancement task so students can experiment instead of just copy code.

**Good next step**
- ask one more question from each PDF and note which chunk was retrieved

## Recall@k

### Explanation: Recall@k
- Recall@k measures whether at least one correct source appears in the top-k retrieved documents (useful when any one correct passage is sufficient).
- Unlike precision@k, recall focuses on coverage (did we retrieve the correct source at all?), which is important for RAG where one supporting passage can be enough.
- The following cells compute recall@4 for a small question set and print average recall.

In [77]:
docs = retriever.invoke("What is maturity benefit in Sanchay Plus policy?")

for d in docs[:4]:
    print(d.metadata.get("source"))

d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 6 — RAG Optimization + LLM Evaluation + LangSmith\Live\HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf
d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 6 — RAG Optimization + LLM Evaluation + LangSmith\Live\HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf
d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 6 — RAG Optimization + LLM Evaluation + LangSmith\Live\HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf
d:\Mentoring\learwithsarvesh\AI Engineer Ready\WEEK 6 — RAG Optimization + LLM Evaluation + LangSmith\Live\HDFC-Life-Sampoorna-Jeevan-101N158V04-Policy-Document (1).pdf


In [83]:
import os

def recall_at_k(retriever, question, gold_sources, k=4):
    docs = retriever.invoke(question)
    topk = docs[:k]
    
    retrieved_sources = [
        os.path.basename(d.metadata.get("source", "")).lower()
        for d in topk
    ]
    
    for gold in gold_sources:
        gold = gold.lower()
        if any(gold in src for src in retrieved_sources):
            return 1.0
    
    return 0.0

### Explanation: `recall_at_k` function
- Calls the retriever to get top-k documents and extracts their source filenames (lowercased).
- Returns `1.0` if any gold source substring matches one of the retrieved filenames, otherwise `0.0`.
- Use recall when you care whether at least one correct passage is present in the top-k results.

In [89]:
questions = [
    {
        "question": "What is the free look period in group term policy?",
        "gold_sources": ["HDFC-Life-Group-Term-Life-Policy.pdf"]
    },
    {
        "question": "What is maturity benefit in Sanchay Plus policy?",
        "gold_sources": ["HDFC-Life-Sanchay-Plus-Life-Long-Income-Option-101N134V19-Policy-Document.pdf"]
    }
]

In [90]:
recalls = []

for q in questions:
    score = recall_at_k(retriever, q["question"], q["gold_sources"], k=4)
    print(f"Recall@4: {score}")
    recalls.append(score)

print("Average Recall@4:", sum(recalls)/len(recalls))

Recall@4: 0.0
Recall@4: 0.0
Average Recall@4: 0.0


### Query Rewriting

### Explanation: Query Rewriting section
- Query rewriting uses an LLM to produce multiple reformatted queries that often retrieve better results.
- The notebook provides helper functions to generate rewrites, retrieve for each rewrite, deduplicate, and return a final top-k list.
- This pipeline can increase recall by rephrasing ambiguous or short user questions.

In [91]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [92]:
def rewrite_query(question):
    prompt = f"""
    You are an expert at improving search queries.

    Given a user question, generate 3 improved versions of it
    to help retrieve better results from documents.

    Question: {question}

    Return only the rewritten queries, one per line.
    """
    
    response = llm.invoke(prompt)
    
    queries = response.content.split("\n")
    queries = [q.strip("- ").strip() for q in queries if q.strip()]
    
    return queries

### Explanation: `rewrite_query` helper
- Takes a short user question and asks an LLM to produce multiple improved query versions.
- The function returns a small list of reformulated queries to run through the retriever.
- Useful when user queries are ambiguous, too short, or missing important qualifiers (dates, entities, policies).

In [93]:
queries = rewrite_query("What is maturity benefit?")
print(queries)

['What are the details and advantages of maturity benefits in insurance policies?', 'Can you explain the concept of maturity benefit and how it works?', 'What is the definition of maturity benefit and what are its implications for policyholders?']


In [94]:
def retrieve_with_rewriting(retriever, question, k=4):
    queries = rewrite_query(question)
    
    print("Generated Queries:")
    for q in queries:
        print("-", q)
    
    all_docs = []
    
    for q in queries:
        docs = retriever.invoke(q)
        all_docs.extend(docs[:k])
    
    return all_docs

### Explanation: `retrieve_with_rewriting`
- Calls `rewrite_query` to generate several alternative queries and runs the retriever for each.
- Aggregates the top-k results from each rewritten query into one list for later deduplication and ranking.
- This broadens the search space and often improves recall for ambiguous questions.

In [95]:
def deduplicate_docs(docs):
    unique = {}
    
    for d in docs:
        key = d.page_content
        unique[key] = d
    
    return list(unique.values())

### Explanation: `deduplicate_docs`
- Removes duplicate chunks by using the chunk's `page_content` as a key.
- Important because query rewriting and hybrid pipelines can return the same passage multiple times, which wastes `k` budget.
- Returns a list of unique Document objects preserving the last seen instance per text key.

In [96]:
def query_rewriting_pipeline(retriever, question, k=4):
    docs = retrieve_with_rewriting(retriever, question, k)
    docs = deduplicate_docs(docs)
    return docs[:k]

### Explanation: `query_rewriting_pipeline`
- End-to-end pipeline that generates rewritten queries, retrieves documents for each rewrite, deduplicates them, and returns the top-k unique documents.
- Use this function in place of `retriever.invoke(question)` when you want higher recall for difficult queries.
- Note: this pipeline is slower because it performs multiple retrievals and LLM calls.

In [97]:
docs = retriever.invoke(question)

In [98]:
docs = query_rewriting_pipeline(retriever, question)

Generated Queries:
- What does "sum assured on death" mean in the context of Sampoorna Jeevan insurance?
- How is the sum assured calculated for death benefits in Sampoorna Jeevan policy?
- What are the details of the death benefit sum assured in the Sampoorna Jeevan plan?


“Query rewriting helps the retriever understand what the user meant, not just what they typed.”

#### Hybrid Retrieval (vector + keyword)

### Explanation: Hybrid Retrieval (vector + keyword)
- Hybrid retrieval merges semantic (vector) and keyword (BM25) search to combine the strengths of both.
- Vector search finds semantically similar passages; BM25 finds lexically matching passages that may have exact answers or names.
- The notebook demonstrates building both retrievers and combining their results, then deduplicating and returning top-k candidates.

Hybrid Retrieval combines semantic search (embeddings) + keyword search (BM25)

User Query
   ↓
        ┌───────────────┐
        │ Vector Search │
        └───────────────┘
               ↓
        ┌───────────────┐
        │ Keyword Search│
        └───────────────┘
               ↓
          Merge Results
               ↓
            Final Docs

In [99]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

Step 2: Keyword Retriever (BM25)

In [108]:
loader = PyPDFLoader("HDFC-Life-Group-Term-Life-Policy.pdf")
docs = loader.load()

In [111]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

documents = splitter.split_documents(docs)

In [114]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 4

In [117]:
from langchain_community.vectorstores import FAISS

embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


vectorstore = FAISS.from_documents(documents, embedding)

In [119]:
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 4

Step 3: Hybrid Function

In [101]:
def hybrid_retrieve(question, k=4):
    vector_docs = vector_retriever.invoke(question)
    keyword_docs = bm25_retriever.invoke(question)
    
    # Combine
    all_docs = vector_docs + keyword_docs
    
    return all_docs

### Explanation: `hybrid_retrieve`
- Calls both the vector retriever (`vector_retriever`) and keyword retriever (`bm25_retriever`) and concatenates their results.
- You can apply weighting or re-ranking after combining results; this simple version returns the raw concatenation for deduplication and downstream re-ranking.
- Later you can implement alpha-weighted fusion to balance lexical and semantic scores.

Deduplicate (VERY IMPORTANT)

In [102]:
def deduplicate_docs(docs):
    unique = {}
    
    for d in docs:
        key = d.page_content
        unique[key] = d
    
    return list(unique.values())

Step 5: Final Hybrid Pipeline

In [106]:
def hybrid_pipeline(question, k=4):
    docs = hybrid_retrieve(question, k)
    docs = deduplicate_docs(docs)
    return docs[:k]

### Explanation: `hybrid_pipeline`
- High-level wrapper that calls `hybrid_retrieve`, deduplicates the results, and returns the final top-k documents.
- Use `hybrid_pipeline(question)` in evaluations to compare hybrid retrieval against pure vector retrieval.
- Because results come from two systems, always deduplicate and consider a re-ranking stage for better precision.

Plug into YOUR evaluation

In [118]:
docs = hybrid_pipeline(question)